<a href="https://colab.research.google.com/github/EfrainHernandezCortes/Procesos-Estocasticos/blob/main/M%C3%A9todo_de_Uniformizaci%C3%B3n_para_una_CMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import scipy.linalg as la
import math

Definimos la matriz R con los parámetros iniciales:

In [16]:
R = np.array(
    [
        [0, 2, 3, 0],
        [4, 0, 2, 0],
        [0, 2, 0, 2],
        [1, 0, 3, 0]
    ],
    dtype=float
)

r_i = np.sum(R, axis=1)
r = 6.0  # max(r_i) = 6.0

Contruimos la matriz $\hat{P}$

In [17]:
N = len(R)
P_gorro = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i == j:
            P_gorro[i, j] = 1.0 - (r_i[i] / r)
        else:
            P_gorro[i, j] = R[i, j] / r

Definimos la función para hacer la aproximación del ejercicio 3:

In [18]:
def aproximacion_ejercicio3(t, P_gorro, r):
    term_propuesto = int(np.ceil(r * t + 5 * np.sqrt(r * t)))
    M = max(term_propuesto, 20)

    P_t = np.zeros_like(P_gorro)
    for k in range(M + 1):
        coef = np.exp(-r * t) * ((r * t) ** k) / math.factorial(k)
        P_t += coef * np.linalg.matrix_power(P_hat, k)
    return P_t, M

Definimos la función que implementa el algoritmo de uniformización del ejercicio 4:

In [19]:
def algoritmo_uniformizacion_ejercicio4(t, P_gorro, r, epsilon=1e-5):
    A = np.eye(N)
    B = np.exp(-r * t) * np.eye(N)
    c = np.exp(-r * t)
    sum_c = c
    k = 1

    while sum_c < 1 - epsilon:
        c = c * (r * t) / k
        A = A @ P_hat
        B = B + c * A
        sum_c = sum_c + c
        k = k + 1

    return B, k - 1 # Regresa la matriz y el M

Definimos los tiempos $t=0.5, 1$ y $5$:

In [20]:
tiempos = [0.5, 1.0, 5.0]

Se ejecutan las funciones para visualizar los resultadosm para cada ejercicio:

In [21]:
print("EJERCICIO 3")
P_05, M_05 = aproximacion_ejercicio3(0.5, P_gorro, r)
P_1, M_1 = aproximacion_ejercicio3(1.0, P_gorro, r)
P_5, M_5 = aproximacion_ejercicio3(5.0, P_gorro, r)

print(f"t=0.5 (M={M_05}):\n{np.round(P_05, 5)}")
print(f"t=1.0 (M={M_1}):\n{np.round(P_1, 5)}")
print(f"t=5.0 (M={M_5}):\n{np.round(P_5, 5)}")

print("\nCOMPROBACIÓN CHAPMAN-KOLMOGOROV")
P_05_cuadrado = P_05 @ P_05
print(f"P(0.5) * P(0.5) =\n{np.round(P_05_cuadrado, 5)}")
print(f"\n¿Se verifica la ecuación de Chapman-Kolmogorov? {np.allclose(P_1, P_05_cuadrado, atol=1e-5)}")

print("\nEJERCICIO 4")
for t in tiempos:
    B_t, M_alg = algoritmo_uniformizacion_ejercicio4(t, P_gorro, r, epsilon=1e-5)
    print(f"t={t} (M necesario={M_alg}):\n{np.round(B_t, 5)}")

EJERCICIO 3
t=0.5 (M=20):
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]
t=1.0 (M=20):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]
t=5.0 (M=58):
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]

COMPROBACIÓN CHAPMAN-KOLMOGOROV
P(0.5) * P(0.5) =
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

¿Se verifica la ecuación de Chapman-Kolmogorov? True

EJERCICIO 4
t=0.5 (M necesario=13):
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]
t=1.0 (M necesario=19):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]